# Drug Court Decision Support System — Decision Tree

This notebook keeps the supplied Decision Tree implementation and adds the output visualizations/tables corresponding to the reference image:
- Confusion Matrix
- Top 10 Global SHAP Feature Importance
- Local SHAP Explanation
- Subgroup Performance by Race
- Subgroup Performance by Gender
- Sample System/Audit Logs

**Model Performance Comparison has been removed.**

All metrics and visualizations are generated from `nijdataset.xlsx` when the notebook is executed; the reference image values are not hard-coded.


In [ ]:
import pandas as pd

from sklearn.model_selection import train_test_split
from sklearn.compose import ColumnTransformer
from sklearn.preprocessing import OneHotEncoder
from sklearn.pipeline import Pipeline
from sklearn.impute import SimpleImputer

from sklearn.tree import DecisionTreeClassifier
from sklearn.metrics import accuracy_score, classification_report, confusion_matrix


In [ ]:
# ============================================================


In [ ]:
# 1. LOAD DATASET


In [ ]:
# ============================================================

df = pd.read_excel("nijdataset.xlsx")

print("Dataset shape:")
print(df.shape)


In [ ]:
# ============================================================


In [ ]:
# 2. FEATURES AND TARGET


In [ ]:
# ============================================================

target = "Recidivism_Within_3years"

features = [
    "Gender",
    "Race",
    "Age_at_Release",
    "Residence_PUMA",
    "Gang_Affiliated",
    "Supervision_Risk_Score_First",
    "Supervision_Level_First",
    "Education_Level",
    "Dependents",
    "Prison_Offense",
    "Prison_Years",
    "Prior_Arrest_Episodes_Felony",
    "Prior_Arrest_Episodes_Misd",
    "Prior_Arrest_Episodes_Violent",
    "Prior_Arrest_Episodes_Property",
    "Prior_Arrest_Episodes_Drug",
    "Prior_Arrest_Episodes_DVCharges",
    "Prior_Arrest_Episodes_GunCharges",
    "Prior_Conviction_Episodes_Felony",
    "Prior_Conviction_Episodes_Misd",
    "Prior_Conviction_Episodes_Viol",
    "Prior_Conviction_Episodes_Prop",
    "Prior_Conviction_Episodes_Drug",
    "Prior_Revocations_Parole",
    "Prior_Revocations_Probation",
    "Condition_MH_SA",
    "Condition_Cog_Ed",
    "Condition_Other",
]

X = df[features].copy()
y = df[target].copy()


In [ ]:
# ============================================================


In [ ]:
# 3. CONVERT TARGET TO 0 AND 1


In [ ]:
# ============================================================

y = y.map({"No": 0, "Yes": 1})


# Check target
print("\nTarget distribution:")
print(y.value_counts())


In [ ]:
# ============================================================


In [ ]:
# 4. DEFINE CATEGORICAL AND NUMERICAL COLUMNS


In [ ]:
# ============================================================

categorical_features = [
    "Gender",
    "Race",
    "Age_at_Release",
    "Gang_Affiliated",
    "Supervision_Level_First",
    "Education_Level",
    "Dependents",
    "Prison_Offense",
    "Prison_Years",
    "Prior_Arrest_Episodes_Felony",
    "Prior_Arrest_Episodes_Misd",
    "Prior_Arrest_Episodes_Violent",
    "Prior_Arrest_Episodes_Property",
    "Prior_Arrest_Episodes_Drug",
    "Prior_Arrest_Episodes_DVCharges",
    "Prior_Arrest_Episodes_GunCharges",
    "Prior_Conviction_Episodes_Felony",
    "Prior_Conviction_Episodes_Misd",
    "Prior_Conviction_Episodes_Viol",
    "Prior_Conviction_Episodes_Prop",
    "Prior_Conviction_Episodes_Drug",
    "Prior_Revocations_Parole",
    "Prior_Revocations_Probation",
    "Condition_MH_SA",
    "Condition_Cog_Ed",
    "Condition_Other",
]

numerical_features = ["Residence_PUMA", "Supervision_Risk_Score_First"]


In [ ]:
# ============================================================


In [ ]:
# 5. CLEAN DATA TYPES


In [ ]:
# ============================================================

# Convert categorical columns to normal Python strings
for column in categorical_features:
    X[column] = X[column].fillna("Missing").astype(str)

# Convert numerical columns to numbers
for column in numerical_features:
    X[column] = pd.to_numeric(X[column], errors="coerce")


print("\nCategorical columns:")
print(categorical_features)

print("\nNumerical columns:")
print(numerical_features)


In [ ]:
# ============================================================


In [ ]:
# 6. PREPROCESSING


In [ ]:
# ============================================================

categorical_transformer = Pipeline(
    steps=[
        ("imputer", SimpleImputer(strategy="most_frequent")),
        ("onehot", OneHotEncoder(handle_unknown="ignore")),
    ]
)

numerical_transformer = Pipeline(steps=[("imputer", SimpleImputer(strategy="median"))])


preprocessor = ColumnTransformer(
    transformers=[
        ("categorical", categorical_transformer, categorical_features),
        ("numerical", numerical_transformer, numerical_features),
    ]
)


In [ ]:
# ============================================================


In [ ]:
# 7. TRAIN / TEST SPLIT


In [ ]:
# ============================================================

X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42, stratify=y
)

print("\nTraining data:")
print(X_train.shape)

print("\nTesting data:")
print(X_test.shape)


In [ ]:
# ============================================================


In [ ]:
# 8. DECISION TREE MODEL


In [ ]:
# ============================================================

model = Pipeline(
    steps=[
        ("preprocessor", preprocessor),
        ("classifier", DecisionTreeClassifier(random_state=42, max_depth=5)),
    ]
)


In [ ]:
# ============================================================


In [ ]:
# 9. TRAIN MODEL


In [ ]:
# ============================================================

print("\nTraining Decision Tree...")

model.fit(X_train, y_train)

print("Training completed!")


In [ ]:
# ============================================================


In [ ]:
# 10. PREDICTION


In [ ]:
# ============================================================

y_pred = model.predict(X_test)


In [ ]:
# ============================================================


In [ ]:
# 11. ACCURACY


In [ ]:
# ============================================================

accuracy = accuracy_score(y_test, y_pred)

print("\nAccuracy:")
print(accuracy)


In [ ]:
# ============================================================


In [ ]:
# 12. CLASSIFICATION REPORT


In [ ]:
# ============================================================

print("\nClassification Report:")

print(
    classification_report(y_test, y_pred, target_names=["No Recidivism", "Recidivism"])
)


In [ ]:
# ============================================================


In [ ]:
# 13. CONFUSION MATRIX


In [ ]:
# ============================================================

print("\nConfusion Matrix:")

print(confusion_matrix(y_test, y_pred))
# Predict recidivism probability
y_prob = model.predict_proba(X_test)[:, 1]

print("\nFirst 10 Predicted Recidivism Probabilities:")

for i in range(10):
    print(f"Person {i+1}: {y_prob[i] * 100:.2f}%")
# pyrefly: ignore [missing-import]
import shap

# Get the preprocessing step
preprocessor_fitted = model.named_steps["preprocessor"]

# Get the Decision Tree model
dt_model = model.named_steps["classifier"]

# Transform the test data
X_test_transformed = preprocessor_fitted.transform(X_test)

# Convert to numeric dense array
X_test_transformed = X_test_transformed.toarray()
X_test_transformed = X_test_transformed.astype(float)

# Get feature names
feature_names = preprocessor_fitted.get_feature_names_out()

# Create SHAP explainer
explainer = shap.TreeExplainer(dt_model)

# Calculate SHAP values for first 10 people
shap_values = explainer.shap_values(X_test_transformed[:10])

print("\nSHAP calculation completed!")
print("Number of transformed features:", len(feature_names))
# Get SHAP values for Recidivism class
if isinstance(shap_values, list):
    shap_for_recidivism = shap_values[1]
else:
    if len(shap_values.shape) == 3:
        shap_for_recidivism = shap_values[:, :, 1]
    else:
        shap_for_recidivism = shap_values

# Calculate average importance
mean_shap = abs(shap_for_recidivism).mean(axis=0)

# Create feature importance table
importance_df = pd.DataFrame({"Feature": feature_names, "Importance": mean_shap})

# Sort from most important to least important
importance_df = importance_df.sort_values(by="Importance", ascending=False)

print("\nTop 15 Important Features:")
print(importance_df.head(15).to_string(index=False))
print("\nIntervention-related columns:")

intervention_features = [
    "Program_Attendances",
    "Program_UnexcusedAbsences",
    "Percent_Days_Employed",
    "Jobs_Per_Year",
    "DrugTests_THC_Positive",
    "DrugTests_Cocaine_Positive",
    "Violations_ElectronicMonitoring",
    "Violations_Instruction",
    "Violations_FailToReport",
]

for column in intervention_features:
    if column in df.columns:
        print(f"\n{column}:")
        print(df[column].head(5).to_list())
    else:
        print(f"\n{column}: column not found in dataset")


In [ ]:
# ==========================================


In [ ]:
# STEP 6: SCENARIO SIMULATION MODEL


In [ ]:
# ==========================================

simulation_features = features + [
    "Program_Attendances",
    "Program_UnexcusedAbsences",
    "Percent_Days_Employed",
    "Jobs_Per_Year",
]

X_sim = df[simulation_features].copy()

# Target
y_sim = df[target].copy()
y_sim = y_sim.map({"No": 0, "Yes": 1})

# Categorical columns
simulation_categorical = categorical_features.copy()

# Numerical columns
simulation_numerical = numerical_features.copy()

# Add the 4 simulation variables as numerical
simulation_numerical += [
    "Program_Attendances",
    "Program_UnexcusedAbsences",
    "Percent_Days_Employed",
    "Jobs_Per_Year",
]

# Convert categorical columns
for column in simulation_categorical:
    X_sim[column] = X_sim[column].fillna("Missing").astype(str)

# Convert numerical columns
for column in simulation_numerical:
    X_sim[column] = pd.to_numeric(X_sim[column], errors="coerce")

# Preprocessing
simulation_categorical_transformer = Pipeline(
    steps=[
        ("imputer", SimpleImputer(strategy="most_frequent")),
        ("onehot", OneHotEncoder(handle_unknown="ignore")),
    ]
)

simulation_numerical_transformer = Pipeline(
    steps=[("imputer", SimpleImputer(strategy="median"))]
)

simulation_preprocessor = ColumnTransformer(
    transformers=[
        ("categorical", simulation_categorical_transformer, simulation_categorical),
        ("numerical", simulation_numerical_transformer, simulation_numerical),
    ]
)

# Train-test split
X_sim_train, X_sim_test, y_sim_train, y_sim_test = train_test_split(
    X_sim, y_sim, test_size=0.2, random_state=42, stratify=y_sim
)

# Decision Tree simulation model
simulation_model = Pipeline(
    steps=[
        ("preprocessor", simulation_preprocessor),
        ("classifier", DecisionTreeClassifier(random_state=42, max_depth=5)),
    ]
)

# Train
simulation_model.fit(X_sim_train, y_sim_train)

# Prediction
y_sim_pred = simulation_model.predict(X_sim_test)

# Accuracy
simulation_accuracy = accuracy_score(y_sim_test, y_sim_pred)

print("\n================================")
print("SCENARIO SIMULATION MODEL")
print("================================")

print("\nSimulation Model Accuracy:")
print(simulation_accuracy)

print("\nClassification Report:")
print(
    classification_report(
        y_sim_test, y_sim_pred, target_names=["No Recidivism", "Recidivism"]
    )
)

print("\nConfusion Matrix:")
print(confusion_matrix(y_sim_test, y_sim_pred))

# Probability
y_sim_prob = simulation_model.predict_proba(X_sim_test)[:, 1]

print("\nFirst 10 Simulation Model Probabilities:")

for i in range(10):
    print(f"Person {i+1}: " f"{y_sim_prob[i] * 100:.2f}%")


In [ ]:
# ==========================================


In [ ]:
# STEP 7: COUNTERFACTUAL SIMULATION


In [ ]:
# ==========================================

# Select one person from the test data
person = X_sim_test.iloc[[0]].copy()

print("\n================================")
print("COUNTERFACTUAL SIMULATION")
print("================================")

print("\nOriginal values:")

print("Program Attendances:", person["Program_Attendances"].iloc[0])

print("Program Unexcused Absences:", person["Program_UnexcusedAbsences"].iloc[0])

print("Percent Days Employed:", person["Percent_Days_Employed"].iloc[0])

print("Jobs Per Year:", person["Jobs_Per_Year"].iloc[0])


# ------------------------------------------
# Current scenario
# ------------------------------------------

current_risk = simulation_model.predict_proba(person)[0][1]

print("\nCurrent Predicted Risk:")
print(f"{current_risk * 100:.2f}%")


# ------------------------------------------
# Hypothetical scenario
# ------------------------------------------

counterfactual_person = person.copy()

counterfactual_person["Program_Attendances"] = 6
counterfactual_person["Program_UnexcusedAbsences"] = 0
counterfactual_person["Percent_Days_Employed"] = 0.50
counterfactual_person["Jobs_Per_Year"] = 1.0


counterfactual_risk = simulation_model.predict_proba(counterfactual_person)[0][1]

print("\nHypothetical Scenario:")
print("Program Attendances: 6")
print("Program Unexcused Absences: 0")
print("Percent Days Employed: 50%")
print("Jobs Per Year: 1")


print("\nHypothetical Predicted Risk:")
print(f"{counterfactual_risk * 100:.2f}%")


# ------------------------------------------
# Compare
# ------------------------------------------

risk_difference = counterfactual_risk - current_risk

print("\nChange in Predicted Risk:")
print(f"{risk_difference * 100:.2f} percentage points")


In [ ]:
# ==========================================


In [ ]:
# STEP 8: MULTIPLE COUNTERFACTUAL SCENARIOS


In [ ]:
# ==========================================

print("\n================================")
print("MULTIPLE SCENARIO SIMULATION")
print("================================")

# Same person used in Step 7
person = X_sim_test.iloc[[0]].copy()


# ------------------------------------------
# Scenario 1: Current situation
# ------------------------------------------

scenario1 = person.copy()


# ------------------------------------------
# Scenario 2: Increase program attendance
# ------------------------------------------

scenario2 = person.copy()

scenario2["Program_Attendances"] = 6


# ------------------------------------------
# Scenario 3: Attendance + employment
# ------------------------------------------

scenario3 = person.copy()

scenario3["Program_Attendances"] = 6
scenario3["Percent_Days_Employed"] = 0.50


# ------------------------------------------
# Scenario 4: Attendance + employment
# + fewer unexcused absences
# ------------------------------------------

scenario4 = person.copy()

scenario4["Program_Attendances"] = 6
scenario4["Percent_Days_Employed"] = 0.50
scenario4["Program_UnexcusedAbsences"] = 0


# ------------------------------------------
# Calculate predicted risks
# ------------------------------------------

risk1 = simulation_model.predict_proba(scenario1)[0][1]
risk2 = simulation_model.predict_proba(scenario2)[0][1]
risk3 = simulation_model.predict_proba(scenario3)[0][1]
risk4 = simulation_model.predict_proba(scenario4)[0][1]


# ------------------------------------------
# Display results
# ------------------------------------------

print("\nScenario Results:")

print(f"Scenario 1 - Current Situation: " f"{risk1 * 100:.2f}%")

print(f"Scenario 2 - Higher Program Attendance: " f"{risk2 * 100:.2f}%")

print(f"Scenario 3 - Attendance + Employment: " f"{risk3 * 100:.2f}%")

print(
    f"Scenario 4 - Attendance + Employment + "
    f"No Unexcused Absences: "
    f"{risk4 * 100:.2f}%"
)


# ------------------------------------------
# Compare with current scenario
# ------------------------------------------

print("\nChange from Current Scenario:")

print(f"Scenario 2: " f"{(risk2 - risk1) * 100:.2f} percentage points")

print(f"Scenario 3: " f"{(risk3 - risk1) * 100:.2f} percentage points")

print(f"Scenario 4: " f"{(risk4 - risk1) * 100:.2f} percentage points")


In [ ]:
# ==========================================


In [ ]:
# STEP 9: AUTOMATIC SCENARIO RECOMMENDATION


In [ ]:
# ==========================================

print("\n================================")
print("AUTOMATIC SCENARIO RECOMMENDATION")
print("================================")

# Store scenario names and risks
scenario_results = {
    "Current Situation": risk1,
    "Higher Program Attendance": risk2,
    "Attendance + Employment": risk3,
    "Attendance + Employment + No Unexcused Absences": risk4,
}

# Find scenario with lowest predicted risk
recommended_scenario = min(scenario_results, key=scenario_results.get)

recommended_risk = scenario_results[recommended_scenario]

# Calculate change from current situation
recommended_change = (recommended_risk - risk1) * 100


print("\nCurrent Predicted Risk:")
print(f"{risk1 * 100:.2f}%")

print("\nRecommended Scenario for Practitioner Review:")
print(recommended_scenario)

print("\nSimulated Predicted Risk:")
print(f"{recommended_risk * 100:.2f}%")

print("\nChange from Current Risk:")
print(f"{recommended_change:.2f} percentage points")

print("\nRecommendation:")
print(
    "This scenario produced the lowest predicted risk "
    "among the simulated scenarios and can be reviewed "
    "by the practitioner."
)

print(
    "\nNote: This is a model-based hypothetical scenario, "
    "not a proven causal intervention effect."
)


In [ ]:
# ==========================================


In [ ]:
# STEP 10: HUMAN-IN-THE-LOOP


In [ ]:
# ============================================================
# STEP 10: HUMAN-IN-THE-LOOP DECISION
# ============================================================

print("\n================================")
print("HUMAN-IN-THE-LOOP DECISION")
print("================================")

print("\nAI Suggested Scenario:")
print(recommended_scenario)
print(f"Simulated Predicted Risk: {recommended_risk * 100:.2f}%")

print("\nPractitioner: choose one of the buttons below.")
print("The notebook will NOT wait for input() and will not get stuck.")

try:
    import ipywidgets as widgets
    from IPython.display import display

    # This variable is used by the following Audit / Feedback Log cell.
    decision = None

    status = widgets.HTML(
        value="<b>Status:</b> Waiting for practitioner decision..."
    )

    accept_button = widgets.Button(
        description="Accept",
        button_style="success",
        tooltip="Accept the AI-suggested scenario"
    )

    modify_button = widgets.Button(
        description="Modify",
        button_style="warning",
        tooltip="Modify the AI-suggested scenario"
    )

    reject_button = widgets.Button(
        description="Reject",
        button_style="danger",
        tooltip="Reject the AI-suggested scenario"
    )

    def set_decision(selected_decision):
        global decision
        decision = selected_decision

        status.value = (
            f"<b>Practitioner Decision:</b> {decision}<br>"
            f"<b>Selected Scenario:</b> {recommended_scenario}<br>"
            f"<b>Simulated Risk:</b> {recommended_risk * 100:.2f}%"
        )

        accept_button.disabled = True
        modify_button.disabled = True
        reject_button.disabled = True

        print(f"\nPractitioner Decision: {decision.upper()}")
        print("\nFinal Decision Status:")
        print(decision)
        print("\nNow run the next AUDIT / FEEDBACK LOG cell.")

    def on_accept(_):
        set_decision("Accepted")

    def on_modify(_):
        set_decision("Modified")

    def on_reject(_):
        set_decision("Rejected")

    accept_button.on_click(on_accept)
    modify_button.on_click(on_modify)
    reject_button.on_click(on_reject)

    display(
        widgets.VBox([
            widgets.HTML("<b>Practitioner Options</b>"),
            widgets.HBox([
                accept_button,
                modify_button,
                reject_button
            ]),
            status
        ])
    )

except ImportError:
    # Non-blocking fallback if ipywidgets is unavailable.
    print("\nipywidgets is not available in this environment.")
    print("Using a non-blocking default decision: Accepted")
    decision = "Accepted"

    print("\nPractitioner Decision: ACCEPTED")
    print("\nFinal Decision Status:")
    print(decision)
    print("\nRun the next AUDIT / FEEDBACK LOG cell.")


In [ ]:
# ==========================================


In [ ]:
# STEP 11: AUDIT / FEEDBACK LOG


In [ ]:
# ==========================================

from datetime import datetime
import csv
import os

# Ensure a practitioner decision was selected before writing the audit record.
if decision is None:
    raise RuntimeError(
        "No practitioner decision selected. Return to the Human-in-the-Loop cell "
        "and click Accept, Modify, or Reject, then run this audit cell again."
    )
print("\n================================")
print("AUDIT / FEEDBACK LOG")
print("================================")

log_dir = "logs"
os.makedirs(log_dir, exist_ok=True)
log_file = os.path.join(log_dir, "decision_log.csv")

# Create log file if it does not exist
file_exists = os.path.exists(log_file)

with open(log_file, "a", newline="") as file:

    writer = csv.writer(file)

    # Write header only once
    if not file_exists:
        writer.writerow(
            [
                "Date_Time",
                "Model_Type",
                "Current_Risk",
                "Recommended_Scenario",
                "Simulated_Risk",
                "Risk_Change",
                "Practitioner_Decision",
            ]
        )

    # Write decision
    writer.writerow(
        [
            datetime.now().strftime("%Y-%m-%d %H:%M:%S"),
            "Decision Tree",
            f"{risk1 * 100:.2f}%",
            recommended_scenario,
            f"{recommended_risk * 100:.2f}%",
            f"{recommended_change:.2f} percentage points",
            decision,
        ]
    )

print("\nDecision saved successfully!")

print("Audit file:")
print(log_file)


In [ ]:
# ==========================================


In [ ]:
# STEP 13A: SAVE MODELS


In [ ]:
# ==========================================

import joblib

model_dir = "saved_models"
os.makedirs(model_dir, exist_ok=True)

baseline_path = os.path.join(model_dir, "decision_tree_baseline_model.pkl")
simulation_path = os.path.join(model_dir, "decision_tree_simulation_model.pkl")

# Save baseline model
joblib.dump(model, baseline_path)

# Save scenario simulation model
joblib.dump(simulation_model, simulation_path)

print("\n================================")
print("MODELS SAVED")
print("================================")

print(f"Baseline model: {baseline_path}")
print(f"Simulation model: {simulation_path}")


In [ ]:
# ============================================================
# REFERENCE OUTPUT 1: CONFUSION MATRIX
# ============================================================

import matplotlib.pyplot as plt
import numpy as np

cm = confusion_matrix(y_test, y_pred)

fig, ax = plt.subplots(figsize=(6, 5))
im = ax.imshow(cm, interpolation="nearest", cmap="YlGnBu")
ax.figure.colorbar(im, ax=ax)

ax.set(
    xticks=np.arange(2),
    yticks=np.arange(2),
    xticklabels=["No Recidivism", "Recidivism"],
    yticklabels=["No Recidivism", "Recidivism"],
    ylabel="True Label",
    xlabel="Predicted Label",
    title="Confusion Matrix — Decision Tree"
)

threshold = cm.max() / 2.0
for i in range(cm.shape[0]):
    for j in range(cm.shape[1]):
        ax.text(
            j, i, str(cm[i, j]),
            ha="center",
            va="center",
            color="white" if cm[i, j] > threshold else "black",
            fontsize=14,
            fontweight="bold"
        )

plt.tight_layout()
plt.show()

print("Confusion Matrix:")
print(cm)


In [ ]:
# ============================================================
# REFERENCE OUTPUT 2: TOP 10 GLOBAL SHAP FEATURE IMPORTANCE
# ============================================================

top10_shap = importance_df.head(10).copy()
top10_shap = top10_shap.sort_values("Importance", ascending=True)

fig, ax = plt.subplots(figsize=(8, 5))
ax.barh(top10_shap["Feature"], top10_shap["Importance"])
ax.set_xlabel("Mean |SHAP value|")
ax.set_ylabel("")
ax.set_title("Top 10 Global Feature Importance (mean SHAP value)")
plt.tight_layout()
plt.show()

display(importance_df.head(10).reset_index(drop=True))


In [ ]:
# ============================================================
# REFERENCE OUTPUT 3: LOCAL SHAP EXPLANATION
# ============================================================

# Explain the first test participant, similar to the reference panel.
local_index = 0
local_transformed = X_test_transformed[local_index]

# Handle SHAP's different return formats across versions.
if isinstance(shap_values, list):
    local_shap = np.asarray(shap_values[-1])[local_index]
else:
    sv = np.asarray(shap_values)
    if sv.ndim == 3:
        local_shap = sv[local_index, :, -1]
    else:
        local_shap = sv[local_index]

base_value = explainer.expected_value
if isinstance(base_value, (list, np.ndarray)):
    base_value = np.asarray(base_value).reshape(-1)[-1]

# Find an ID if the dataset has one; otherwise use a test-row label.
id_candidates = [
    "ID", "Id", "id", "Participant_ID", "participant_id",
    "Person_ID", "person_id", "Case_ID", "case_id"
]

participant_label = None
for col in id_candidates:
    if col in df.columns:
        participant_label = str(df.loc[X_test.index[local_index], col])
        break

if participant_label is None or participant_label == "nan":
    participant_label = f"Test-{local_index + 1:03d}"

print(f"SHAP Local Explanation — {participant_label}")

try:
    local_explanation = shap.Explanation(
        values=local_shap,
        base_values=float(base_value),
        data=local_transformed,
        feature_names=feature_names
    )
    shap.plots.waterfall(
        local_explanation,
        max_display=10,
        show=False
    )
    plt.title(f"SHAP Local Explanation — {participant_label}")
    plt.tight_layout()
    plt.show()
except Exception:
    # Fallback for older SHAP versions.
    local_df = pd.DataFrame({
        "Feature": feature_names,
        "SHAP_Value": local_shap
    })
    local_df["Abs_SHAP"] = local_df["SHAP_Value"].abs()
    local_df = local_df.sort_values(
        "Abs_SHAP", ascending=False
    ).head(10).sort_values("SHAP_Value")

    plt.figure(figsize=(8, 5))
    plt.barh(local_df["Feature"], local_df["SHAP_Value"])
    plt.axvline(0, linewidth=1)
    plt.xlabel("SHAP value")
    plt.title(f"SHAP Local Explanation — {participant_label}")
    plt.tight_layout()
    plt.show()

local_table = pd.DataFrame({
    "Feature": feature_names,
    "SHAP_Value": local_shap
})
local_table["Abs_SHAP"] = local_table["SHAP_Value"].abs()

print("\nTop local contributing features:")
display(
    local_table
    .sort_values("Abs_SHAP", ascending=False)
    .head(10)
    .drop(columns="Abs_SHAP")
    .reset_index(drop=True)
)


In [ ]:
# ============================================================
# REFERENCE OUTPUT 4: SUBGROUP PERFORMANCE BY RACE
# ============================================================

from sklearn.metrics import precision_score, recall_score, f1_score

def subgroup_metrics(y_true_group, y_pred_group):
    if len(y_true_group) == 0:
        return {
            "accuracy": np.nan,
            "precision": np.nan,
            "recall": np.nan,
            "fpr": np.nan,
            "f1": np.nan,
        }

    tn, fp, fn, tp = confusion_matrix(
        y_true_group,
        y_pred_group,
        labels=[0, 1]
    ).ravel()

    return {
        "accuracy": accuracy_score(y_true_group, y_pred_group),
        "precision": precision_score(
            y_true_group, y_pred_group, zero_division=0
        ),
        "recall": recall_score(
            y_true_group, y_pred_group, zero_division=0
        ),
        "fpr": fp / (fp + tn) if (fp + tn) else np.nan,
        "f1": f1_score(
            y_true_group, y_pred_group, zero_division=0
        ),
    }

race_results = []

if "Race" in X_test.columns:
    race_values = X_test["Race"].fillna("Missing").astype(str)

    for group in sorted(race_values.unique()):
        mask = race_values == group
        metrics = subgroup_metrics(
            y_test[mask],
            y_pred[mask]
        )

        race_results.append({
            "group": group,
            "n": int(mask.sum()),
            **metrics
        })

race_df = pd.DataFrame(race_results)

print("Subgroup performance by RACE")

if not race_df.empty:
    display(
        race_df.style.format({
            "accuracy": "{:.3f}",
            "precision": "{:.3f}",
            "recall": "{:.3f}",
            "fpr": "{:.3f}",
            "f1": "{:.3f}"
        })
    )
else:
    print("Race column is not available.")


In [ ]:
# ============================================================
# REFERENCE OUTPUT 5: SUBGROUP PERFORMANCE BY GENDER
# ============================================================

gender_results = []

if "Gender" in X_test.columns:
    gender_values = X_test["Gender"].fillna("Missing").astype(str)

    for group in sorted(gender_values.unique()):
        mask = gender_values == group
        metrics = subgroup_metrics(
            y_test[mask],
            y_pred[mask]
        )

        gender_results.append({
            "group": group,
            "n": int(mask.sum()),
            **metrics
        })

gender_df = pd.DataFrame(gender_results)

print("Subgroup performance by GENDER")

if not gender_df.empty:
    display(
        gender_df.style.format({
            "accuracy": "{:.3f}",
            "precision": "{:.3f}",
            "recall": "{:.3f}",
            "fpr": "{:.3f}",
            "f1": "{:.3f}"
        })
    )
else:
    print("Gender column is not available.")


In [ ]:
# ============================================================
# REFERENCE OUTPUT 6: SAMPLE SYSTEM LOGS
# ============================================================

# If the audit log exists, show its latest five records.
# Otherwise, display a compact sample log in the same structure
# as the reference image.

if os.path.exists(log_file):
    logs_df = pd.read_csv(log_file)
    display(logs_df.tail(5))
else:
    sample_logs = pd.DataFrame([
        {
            "timestamp": datetime.now().strftime("%Y-%m-%d %H:%M:%S"),
            "user": "drug_court_system",
            "role": "JUDGE",
            "action": "LOGIN",
            "participant_id": participant_label,
            "status": "success"
        },
        {
            "timestamp": datetime.now().strftime("%Y-%m-%d %H:%M:%S"),
            "user": "drug_court_system",
            "role": "JUDGE",
            "action": "ASSESSMENT_GENERATED",
            "participant_id": participant_label,
            "status": "success"
        },
        {
            "timestamp": datetime.now().strftime("%Y-%m-%d %H:%M:%S"),
            "user": "drug_court_system",
            "role": "JUDGE",
            "action": "SHAP_EXPLANATION_GENERATED",
            "participant_id": participant_label,
            "status": "success"
        },
        {
            "timestamp": datetime.now().strftime("%Y-%m-%d %H:%M:%S"),
            "user": "drug_court_system",
            "role": "JUDGE",
            "action": "SCENARIO_SIMULATION",
            "participant_id": participant_label,
            "status": "success"
        },
        {
            "timestamp": datetime.now().strftime("%Y-%m-%d %H:%M:%S"),
            "user": "drug_court_system",
            "role": "JUDGE",
            "action": "RECOMMENDATION_GENERATED",
            "participant_id": participant_label,
            "status": "success"
        }
    ])

    display(sample_logs)


In [ ]:
# ============================================================
# SAVE OUTPUT TABLES
# ============================================================

output_dir = "outputs"
os.makedirs(output_dir, exist_ok=True)

importance_df.head(15).to_csv(
    os.path.join(output_dir, "global_shap_feature_importance.csv"),
    index=False
)
race_df.to_csv(
    os.path.join(output_dir, "subgroup_performance_race.csv"),
    index=False
)
gender_df.to_csv(
    os.path.join(output_dir, "subgroup_performance_gender.csv"),
    index=False
)

print("Output tables saved in:", output_dir)
